In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pytensor

floatX = "float32"
pytensor.config.floatX = floatX

In [ ]:
import numpy as np

from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

from pytensor_ml.activations import LeakyReLU
from pytensor_ml.layers import BatchNorm2D, Dropout, Input, Linear, Sequential
from pytensor_ml.loss import CrossEntropy
from pytensor_ml.model import Model
from pytensor_ml.optim import adagrad
from pytensor_ml.util import DataLoader

In [ ]:
X, y = load_digits(return_X_y=True)
y_onehot = OneHotEncoder().fit_transform(y[:, None]).toarray().astype(floatX)
X_normed = MinMaxScaler().fit_transform(X).astype(floatX)

In [ ]:
X_in = Input("X_in", shape=(None, 64))

In [ ]:
prediction_network = Sequential(
    Linear("Linear_1", n_in=64, n_out=256),
    BatchNorm2D("BatchNorm_1"),
    Dropout("Dropout_1", p=0.5),
    LeakyReLU(),
    Linear("Linear_2", n_in=256, n_out=128),
    BatchNorm2D("BatchNorm_2"),
    Dropout("Dropout_2", p=0.5),
    LeakyReLU(),
    Linear("Logits", n_in=128, n_out=10),
)

y_hat = prediction_network(X_in)
model = Model(X_in, y_hat, compile_kwargs={"mode": "NUMBA"})
model

In [ ]:
loss_fn = CrossEntropy(expect_onehot_labels=True, expect_logits=True, reduction="mean")

In [ ]:
# `model.compile_train` is the training companion to `model.predict`: it builds the loss against a target,
# differentiates it, folds in the BatchNorm running-statistic updates, and compiles a one-step training
# function (reusing the model's NUMBA compile settings).
step = model.compile_train(adagrad(learning_rate=1e-3), loss_fn, ndim_out=2)

In [ ]:
model.initialize()

In [ ]:
from tqdm.notebook import tqdm

loader = DataLoader(X_normed, y_onehot)
loss_history = []
n_epochs = 10_000

for _ in tqdm(range(n_epochs)):
    X_batch, y_batch = loader()
    loss_value = step(X_batch, y_batch)
    loss_history.append(loss_value)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(loss_history)

In [ ]:
from scipy.special import softmax

# `model.predict` compiles a forward pass specialized for inference: Dropout is dropped and BatchNorm uses its
# running statistics.
y_hat_logits = model.predict(X_normed)
y_hat_probs = softmax(y_hat_logits, axis=-1)
y_pred = np.argmax(y_hat_probs, axis=-1)

In [ ]:
import seaborn as sns

from sklearn.metrics import confusion_matrix

sns.heatmap(confusion_matrix(y, y_pred), annot=True, fmt="0.0f")